<a href="https://colab.research.google.com/github/chauhan-megha/Internship-final-project-/blob/main/Internship_python_project_on_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np



In [2]:
# 2. Load dataset
df = pd.read_csv("synthetic datasets.csv")

In [3]:
# Check dataset
print(df.head())
print(df.info())
print(df.isnull().sum())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 co

In [4]:
print(df["isFraud"].value_counts())

fraud_percentage = df["isFraud"].mean() * 100
print("Fraud Percentage:", fraud_percentage)

isFraud
0    1047433
1       1142
Name: count, dtype: int64
Fraud Percentage: 0.10890971079798775


In [5]:
df = pd.get_dummies(df, columns=["type"], drop_first=True)

In [7]:
df["orig_balance_change"] = df["oldbalanceOrg"] - df["newbalanceOrig"]

df["dest_balance_change"] = df["newbalanceDest"] - df["oldbalanceDest"]

df["amount_to_balance_ratio"] = (
    df["amount"] / (df["oldbalanceOrg"] + 1)
)

In [8]:
X = df.drop(columns=["isFraud"])

y = df["isFraud"]

In [9]:
X = X.select_dtypes(include=["number", "bool"])

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [12]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.999971389743223
Precision: 1.0
Recall: 0.9736842105263158
F1 Score: 0.9866666666666667
ROC-AUC: 0.9890076606520575

Confusion Matrix:
[[209487      0]
 [     6    222]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    209487
           1       1.00      0.97      0.99       228

    accuracy                           1.00    209715
   macro avg       1.00      0.99      0.99    209715
weighted avg       1.00      1.00      1.00    209715



In [15]:
X_test_result = X_test.copy()

X_test_result["Actual_Fraud"] = y_test.values
X_test_result["Predicted_Fraud"] = y_pred
X_test_result["Fraud_Probability"] = y_prob

X_test_result.to_csv(
    "fraud_prediction_results.csv",
    index=False
)